In [1]:
import os
import requests
from IPython.display import Markdown, display, update_display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
from faster_whisper import WhisperModel
from dotenv import load_dotenv
import gradio as gr


In [2]:
load_dotenv()
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_size = "medium"
model=WhisperModel(model_size, device="cuda", compute_type="int8")


In [4]:
def llm_config():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )
    return quant_config

In [5]:
def transribe_audio(model,audioFile):
    segments, info = model.transcribe(audioFile, beam_size=5)
    return " ".join([segment.text for segment in segments])

In [6]:
def prompt(transcript):
  system_message = "You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown."
  user_prompt = f"Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.\n{transcript}"

  messages = [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_prompt}
    ]
  return messages

In [7]:
def transcribe_summarise(audioFile,quant_config,messages):    
    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer)
    model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
    outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [8]:
def generate_minutes(audioFile):
    transcript = transribe_audio(model,audioFile)
    messages = prompt(transcript)
    quant_config = llm_config()
    mom=transcribe_summarise(audioFile,quant_config,messages)
    return mom
    


In [9]:
generate_minutes("denver_extract.mp3")

OSError: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like meta-llama/Meta-Llama-3.1-8B-Instruct is not the path to a directory containing a file named config.json.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
# demo=gr.Interface(
#     fn=generate_minutes,
#     inputs=gr.Audio(type="filepath",label="Upload Audio File"),
#     outputs=gr.Textbox(label="Minutes of Meeting",lines=40, interactive=False),
#     title="Meeting Minutes Generator",
#     description="This app generates minutes of meetings from audio recordings.",
#     theme="default",
#     css=".output_textbox { height: 400px; }"
# )
# if __name__ == "__main__":
#     demo.launch(share=True, debug=True)
